# Случайность в играх: практикум на C#

**Тема 8. Методы генерации случайных чисел и метод Монте-Карло**

Почти в каждой игре есть случайность: что выпадет из сундука, попадёт ли удар, где появится враг, какой будет карта уровня. В этой тетрадке вы разберёте, **как компьютер получает случайные числа** и **как из них делают игровые механики**, — и всё напишете сами на C#: генераторы, броски кубиков, таблицы лута, перемешивание колоды, криты, гарант на легендарку, генерацию пещер. В конце — итоговая задача: настроить шансы в мини-игре «Рыбалка» методом Монте-Карло.

### Как устроена тетрадка

Каждый уровень состоит из одинаковых блоков:

| Блок | Что делать |
|---|---|
| **Теория** | Прочитать |
| **Пример** | Запустить ячейку и разобрать результат |
| **Задание** | Дописать код вместо `throw new NotImplementedException();` или строк `// TODO` |
| **Проверка** | Запустить: она сама покажет, всё ли верно |
| **Вопросы** | Ответить письменно в ячейке под вопросом |
| **В геймдеве** | Где это встречается в настоящих играх |

### Что пишет проверка

- `[OK]` — тест пройден;
- `[ОШИБКА]` — результат неверный, ниже будет подсказка;
- `[ПУСТО]` — функция ещё не написана;
- `[СБОЙ]` — программа упала с исключением (текст ошибки будет рядом).

### Правила

1. Запускайте ячейки **по порядку**: `Shift + Enter` — выполнить ячейку и перейти к следующей.
2. Если перезапустили ядро (Restart) — выполните все ячейки сверху заново (*Run All Above*).
3. Следующие задания используют функции из предыдущих. Не пропускайте уровни.
4. В конце есть ячейка «Мой прогресс» — она покажет, сколько заданий засчитано.

> **Запуск.** Нужны .NET 9 SDK и ядро .NET Interactive:
> `dotnet tool install -g Microsoft.dotnet-interactive`, затем `dotnet interactive jupyter install`.
> Откройте тетрадку в Jupyter или VS Code и выберите ядро **.NET (C#)**.

---
## Уровень 0. Подготовка

Запустите ячейку ниже. Она подключает нужные библиотеки и создаёт помощников: функции проверки и рисования гистограмм. Разбираться в ней не обязательно.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;

// ---------- система проверок ----------
var _progress = new Dictionary<string, bool>();
string _currentTask = "";

void Begin(string id, string title)
{
    _currentTask = id;
    _progress[id] = true;
    Console.WriteLine($"Проверка задания {id}: {title}");
}

void Test(string what, Func<bool> test, string hint = "")
{
    bool ok = false;
    try
    {
        ok = test();
        Console.WriteLine(ok ? $"  [OK]     {what}" : $"  [ОШИБКА] {what}");
        if (!ok && hint != "") Console.WriteLine($"           Подсказка: {hint}");
    }
    catch (NotImplementedException)
    {
        Console.WriteLine($"  [ПУСТО]  {what}: функция ещё не написана");
    }
    catch (Exception e)
    {
        Console.WriteLine($"  [СБОЙ]   {what}: {e.GetType().Name}: {e.Message}");
    }
    if (!ok) _progress[_currentTask] = false;
}

void Done()
{
    Console.WriteLine(_progress[_currentTask]
        ? "Итог: задание засчитано."
        : "Итог: пока не всё. Исправьте функцию, выполните её ячейку и запустите проверку ещё раз.");
}

// ---------- текстовая гистограмма ----------
void Histogram(IEnumerable<(string label, double value)> rows, int width = 40)
{
    var list = rows.ToList();
    double max = list.Max(r => r.value);
    foreach (var (label, value) in list)
    {
        int len = max > 0 ? (int)Math.Round(value / max * width) : 0;
        Console.WriteLine($"{label,14} | {new string('█', len)} {value:0.####}");
    }
}

Console.WriteLine("Готово.");

---
## Уровень 1. Генератор своими руками: LCG

### Теория

Компьютер не умеет «подбрасывать монетку». Есть два способа получить случайность:

- **Аппаратный генератор** (АГСЧ) измеряет физический шум: тепловой шум в микросхеме, радиоактивный распад, атмосферные помехи. Такие числа по-настоящему случайны, но их получают медленно и повторить нельзя. В процессорах Intel и AMD для этого есть инструкция `RDRAND`, а компания Cloudflare снимает на камеру стену из сотни лавовых ламп и превращает картинку в случайные биты.
- **Программный генератор** (ГПСЧ — генератор *псевдо*случайных чисел) берёт число и по **формуле** получает из него следующее, из следующего — ещё одно и так далее:

$$x_{n+1} = f(x_n)$$

Числа выглядят случайными, но полностью определены первым числом — **зерном** (seed). Это быстро, бесплатно и, что важно для игр, **воспроизводимо**: то же зерно — та же последовательность.

### Первый ГПСЧ: метод середины квадрата

В 1946 году Джон фон Нейман, работая над первыми компьютерами, предложил такой способ: возвести 4-значное число в квадрат, дописать слева нули до 8 цифр и взять **средние 4 цифры**. Например, `5772² = 33315984` → следующее число `3159`.

### Пример: метод середины квадрата

In [ ]:
int MiddleSquare(int x)
{
    long square = (long)x * x;              // квадрат: не больше 8 цифр
    return (int)(square / 100 % 10000);     // отрезаем 2 цифры справа и берём 4 следующие
}

foreach (int seed in new[] { 5772, 2916 })
{
    Console.Write($"зерно {seed}: ");
    int v = seed;
    for (int i = 0; i < 10; i++)
    {
        v = MiddleSquare(v);
        Console.Write($"{v} ");
    }
    Console.WriteLine();
}

Метод быстро **вырождается**: от зерна 5772 последовательность за 7 шагов скатывается в ноль и там остаётся, а 2916 через 4 шага возвращается к себе — из 10 000 возможных чисел генератор выдаёт только четыре. Фон Нейман знал об этом, но для расчётов того времени метод годился: ошибку хотя бы было видно сразу.

### Линейный конгруэнтный генератор

В 1949 году Деррик Лемер предложил генератор, который используется до сих пор, — **линейный конгруэнтный** (LCG, *Linear Congruential Generator*):

$$x_{n+1} = (a \cdot x_n + c) \bmod m$$

- `m` — модуль: все числа будут от `0` до `m − 1`;
- `a` — множитель, `c` — приращение;
- `x₀` — зерно.

`mod` — это **остаток от деления**, в C# он пишется `%`. Например, `38 % 16 = 6`, потому что `38 = 2·16 + 6`.

LCG стоит внутри `rand()` во многих реализациях языка C и внутри `java.util.Random` (там `m = 2⁴⁸`). В справочнике *Numerical Recipes* рекомендуют параметры `a = 1664525`, `c = 1013904223`, `m = 2³²`.

**Пример вручную** (`m = 16`, `a = 5`, `c = 3`, `x₀ = 7`):

- `x₁ = (5·7 + 3) % 16 = 38 % 16 = 6`
- `x₂ = (5·6 + 3) % 16 = 33 % 16 = 1`
- `x₃ = (5·1 + 3) % 16 = 8`
- …

### Задание 1.1

Напишите функцию `LcgNext`, которая по текущему числу `x` возвращает следующее.

In [ ]:
int LcgNext(int x, int a, int c, int m)
{
    // TODO: верните следующий член последовательности по формуле LCG
    throw new NotImplementedException();
}

In [ ]:
Begin("1.1", "LcgNext — следующий член LCG");
Test("(5·7 + 3) % 16 = 6", () => LcgNext(7, 5, 3, 16) == 6);
Test("(5·6 + 3) % 16 = 1", () => LcgNext(6, 5, 3, 16) == 1);
Test("вся последовательность для m=16, a=5, c=3, x₀=7", () =>
{
    int[] expected = { 6, 1, 8, 11, 10, 5, 12, 15, 14, 9, 0, 3, 2, 13, 4, 7 };
    int x = 7;
    foreach (int e in expected)
    {
        x = LcgNext(x, 5, 3, 16);
        if (x != e) return false;
    }
    return true;
}, "Сначала умножение и сложение, потом остаток: (a * x + c) % m.");
Done();

### Пример: смотрим на последовательность

Запустите и посмотрите внимательно: после 16 чисел последовательность **повторяется**.

In [ ]:
int lcgX = 7;
for (int i = 1; i <= 34; i++)
{
    lcgX = LcgNext(lcgX, 5, 3, 16);
    Console.Write($"{lcgX} ");
    if (i % 16 == 0) Console.WriteLine("  ← цикл замкнулся");
}

### Период

Чисел всего `m`, поэтому рано или поздно какое-то число повторится — и дальше всё пойдёт по кругу. Длина этого круга называется **периодом**. Хороший генератор имеет период, равный `m` (перебирает все числа).

**Теорема Халла — Добелла.** При `c ≠ 0` период равен `m`, только если:
1. `c` и `m` не имеют общих делителей (кроме 1);
2. `a − 1` делится на **все простые делители** `m`;
3. если `m` делится на 4, то и `a − 1` делится на 4.

### Задание 1.2

Напишите функцию `Period`, которая находит длину цикла. Идея: запоминаем, на каком шаге видели каждое число. Как только число встретилось второй раз — период равен разнице шагов.

In [ ]:
int Period(int seed, int a, int c, int m)
{
    var seen = new Dictionary<int, int>();   // число -> номер шага, когда мы его увидели
    int x = seed;
    int step = 0;

    // TODO:
    // пока x ещё нет в seen (seen.ContainsKey(x) == false):
    //     запомните: seen[x] = step
    //     получите следующее число: x = LcgNext(x, a, c, m)
    //     увеличьте step на 1
    // когда цикл закончился, x встретилось повторно:
    //     верните step - seen[x]

    throw new NotImplementedException();
}

In [ ]:
Begin("1.2", "Period — длина цикла");
Test("m=16, a=5, c=3 → период 16", () => Period(7, 5, 3, 16) == 16);
Test("m=16, a=4, c=1 → период 1 (застряли на одном числе)", () => Period(7, 4, 1, 16) == 1,
     "Последовательность 13, 5, 5, 5… — число 5 повторяется сразу, период 1.");
Test("m=16, a=3, c=0, x₀=1 → период 4", () => Period(1, 3, 0, 16) == 4);
Test("m=9, a=4, c=1 → период 9", () => Period(0, 4, 1, 9) == 9);
Done();

### Эксперимент

Запустите и сравните периоды с теоремой Халла — Добелла. Добавьте свои варианты параметров в список.

In [ ]:
var tries = new (int a, int c, int m)[]
{
    (5, 3, 16), (4, 1, 16), (3, 0, 16),
    (4, 1, 9),  (7, 2, 9),  (2, 1, 9),
    (21, 7, 100), (20, 7, 100),
};
foreach (var (a, c, m) in tries)
    Console.WriteLine($"a = {a,3}, c = {c,2}, m = {m,3}  →  период {Period(0, a, c, m),3} из {m}");

### Вопросы

1. Какие строки эксперимента дали полный период? Проверьте их по трём условиям теоремы.
2. Посмотрите на последовательность `6, 1, 8, 11, 10, 5, 12, …` — чётные и нечётные числа **чередуются**. Почему так происходит при нечётных `a` и `c`? Чем это плохо для игры, если брать `x % 2` как «орёл/решка»?

*Ваш ответ:* …

### Плохой генератор видно глазами

Простой тест качества: построить точки `(xₙ, xₙ₊₁)` — каждое число против следующего. У хорошего генератора точки равномерно заполняют квадрат. У плохого — выстраиваются в линии: зная одно число, можно почти угадать следующее.

Самый известный пример — генератор **RANDU** фирмы IBM (1960-е, `xₙ₊₁ = 65539·xₙ mod 2³¹`). Тройки его чисел лежат всего на 15 плоскостях в кубе. Им пользовались десятилетиями, и результаты многих научных расчётов тех лет пришлось перепроверять.

Сравним маленький LCG (`a = 5, c = 1, m = 256`) и встроенный `System.Random`.

In [ ]:
void PairPlot(string title, IEnumerable<int> numbers, int range)
{
    const int W = 48, H = 20;
    var grid = new char[H, W];
    for (int r = 0; r < H; r++)
        for (int c = 0; c < W; c++)
            grid[r, c] = ' ';

    int prev = -1;
    foreach (int v in numbers)
    {
        if (prev >= 0)
            grid[H - 1 - v * H / range, prev * W / range] = '*';
        prev = v;
    }

    Console.WriteLine(title);
    for (int r = 0; r < H; r++)
    {
        var line = new StringBuilder("|");
        for (int c = 0; c < W; c++) line.Append(grid[r, c]);
        Console.WriteLine(line.ToString());
    }
    Console.WriteLine("+" + new string('-', W) + "\n");
}

var lcgSeq = new List<int>();
int lv = 7;
for (int i = 0; i < 257; i++)
{
    lcgSeq.Add(lv);
    lv = LcgNext(lv, 5, 1, 256);
}
PairPlot("LCG a=5, c=1, m=256: точки (x[n], x[n+1])", lcgSeq, 256);

var sysRng = new Random(3);
PairPlot("System.Random: точки (x[n], x[n+1])", Enumerable.Range(0, 257).Select(_ => sysRng.Next(256)), 256);

У LCG все 256 точек легли на пять прямых. У `System.Random` — облако без структуры.

> **В геймдеве.** Генератор, который зависит только от зерна, — это суперсила: в Minecraft одно число (seed) задаёт весь мир, в мультиплеере всем игрокам достаточно передать зерно, а не карту целиком, а баг с «неудачным» выпадением можно повторить и отладить.

---
## Уровень 2. Xorshift — битовая магия

### Теория

Современные генераторы работают с **битами** числа. Нужны две операции:

**XOR** (исключающее ИЛИ, в C# — `^`): бит результата равен 1, если биты **разные**.

| a | b | a ^ b |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**Сдвиг** `x << k` — все биты уезжают влево на `k` позиций, справа появляются нули. `x >> k` — то же вправо.

Генератор **Xorshift** (Джордж Марсалья, 2003) делает с числом всего три шага «XOR со сдвигом», и биты перемешиваются так, что результат трудно угадать. Процессор выполняет такие операции очень быстро.

### Пример: перемешиваем 8 бит

In [ ]:
string Bits(uint v, int width = 8) => Convert.ToString(v, 2).PadLeft(width, '0');

uint b = 180;
Console.WriteLine($"x                  = {Bits(b)}  ({b})");
uint shifted = (b << 2) & 0xFF;          // & 0xFF — оставляем только 8 младших бит
Console.WriteLine($"x << 2             = {Bits(shifted)}");
b ^= shifted;
Console.WriteLine($"x = x ^ (x << 2)   = {Bits(b)}  ({b})");
uint shifted2 = b >> 3;
Console.WriteLine($"x >> 3             = {Bits(shifted2)}");
b ^= shifted2;
Console.WriteLine($"x = x ^ (x >> 3)   = {Bits(b)}  ({b})");

### Задание 2.1

Напишите 32-битный Xorshift. Он делает три шага подряд:

1. `x` = `x` XOR (`x` сдвинутый **влево на 13**)
2. `x` = `x` XOR (`x` сдвинутый **вправо на 17**)
3. `x` = `x` XOR (`x` сдвинутый **влево на 5**)

Тип `uint` — беззнаковое 32-битное число. Лишние биты при сдвиге влево отбрасываются сами, ничего обрезать не нужно. Подсказка: запись `x ^= y` означает `x = x ^ y`.

In [ ]:
uint Xorshift32(uint x)
{
    // TODO: три шага «XOR со сдвигом»
    throw new NotImplementedException();
}

In [ ]:
Begin("2.1", "Xorshift32");
Test("первые три числа от зерна 2463534242 совпадают с эталоном Марсальи", () =>
{
    uint s = 2463534242;
    uint[] expected = { 723471715, 2497366906, 2064144800 };
    foreach (uint e in expected)
    {
        s = Xorshift32(s);
        if (s != e) return false;
    }
    return true;
}, "Порядок сдвигов: влево 13, вправо 17, влево 5. Каждый раз XOR с самим x.");
Done();

Console.WriteLine($"\nА что будет с зерном 0? Xorshift32(0) = {Xorshift32(0)}");

### Задание 2.2

Генератор выдаёт огромные числа от `0` до `4 294 967 295` (это `2³² − 1`). В играх обычно нужно дробное число от 0 до 1 — из него легко получить что угодно. Напишите `ToUnit`, которая делит `x` на `2³² = 4294967296` и возвращает число из `[0, 1)`.

 Осторожно: если делить целое на целое, C# отбросит дробную часть и получится 0.

In [ ]:
double ToUnit(uint x)
{
    // TODO: верните x, делённое на 2^32, как дробное число
    throw new NotImplementedException();
}

In [ ]:
Begin("2.2", "ToUnit — число из [0, 1)");
Test("ToUnit(0) = 0", () => ToUnit(0) == 0);
Test("ToUnit(2147483648) = 0.5", () => ToUnit(2147483648) == 0.5,
     "Похоже на целочисленное деление. Напишите делитель как 4294967296.0");
Test("ToUnit(uint.MaxValue) чуть меньше 1", () => ToUnit(uint.MaxValue) < 1 && ToUnit(uint.MaxValue) > 0.9999);
Done();

### Пример: свой генератор кубика

Теперь у нас есть собственный генератор! Бросим кубик d6 десять раз: `1 + (int)(U * 6)`, где `U` — число из `[0, 1)`.

In [ ]:
uint state = 2026;   // зерно
for (int i = 0; i < 10; i++)
{
    state = Xorshift32(state);
    int roll = 1 + (int)(ToUnit(state) * 6);
    Console.Write($"{roll} ");
}

### Вопросы

1. Почему зерно Xorshift **не может быть 0**?
2. Почему `1 + (int)(U * 6)` всегда даёт число от 1 до 6 и никогда не даёт 7?

*Ваш ответ:* …

### Теория: что стоит внутри настоящих генераторов

**Вихрь Мерсенна** (Mersenne Twister, М. Мацумото и Т. Нисимура, 1997) — стандарт де-факто последних 25 лет. Его период `2¹⁹⁹³⁷ − 1` — это простое число Мерсенна, отсюда название. Для сравнения: `2¹⁹⁹³⁷ ≈ 10⁶⁰⁰¹`, а атомов в наблюдаемой Вселенной около `10⁸⁰`. Состояние генератора — 624 числа по 32 бита. Он стоит в Python (`random`), в C++ (`std::mt19937`), в Excel (`СЛЧИС`).

**xoshiro / xorshift** — потомки Xorshift из задания выше: быстрее и компактнее вихря Мерсенна при сопоставимом качестве. **PCG** (Мелисса О'Нил, 2014) — LCG, к выходу которого добавлено перемешивание битов; исправляет главный недостаток LCG.

| Где | Алгоритм | Полезно знать |
|---|---|---|
| C#: `System.Random` (.NET 6+) | xoshiro256** | если задать зерно, включается старый алгоритм Кнута |
| Unity: `UnityEngine.Random` | Xorshift128 | один общий генератор на всю игру |
| Godot: `RandomNumberGenerator` | PCG32 | алгоритм может смениться между версиями движка |
| Python: `random` | вихрь Мерсенна | |
| C++: `std::mt19937` | вихрь Мерсенна | `std::rand()` зависит от компилятора, часто LCG |
| Java: `java.util.Random` | LCG, `m = 2⁴⁸` | |
| JavaScript: `Math.random` (Chrome) | xorshift128+ | зерно задать нельзя |

Вывод: свой генератор для игры писать почти никогда не нужно — нужно уметь правильно пользоваться готовым.

**Криптостойкие генераторы.** Все генераторы выше предсказуемы: например, увидев 624 подряд идущих числа вихря Мерсенна, можно восстановить его состояние и предсказать все следующие. Для паролей, токенов и всего, что связано с деньгами, нужен **криптостойкий** генератор — по выданным числам нельзя угадать следующие. Он берёт энтропию из операционной системы (те самые аппаратные источники). В C# это класс `RandomNumberGenerator`.

### Пример: обычный генератор против криптостойкого

In [ ]:
using System.Security.Cryptography;
using System.Diagnostics;

Console.WriteLine($"Криптостойкий бросок d6: {RandomNumberGenerator.GetInt32(1, 7)}");
Console.WriteLine($"Токен сессии: {Convert.ToHexString(RandomNumberGenerator.GetBytes(16))}");

var sw = Stopwatch.StartNew();
var fast = new Random(1);
long sum1 = 0;
for (int i = 0; i < 1_000_000; i++) sum1 += fast.Next(1, 7);
long t1 = sw.ElapsedMilliseconds;

sw.Restart();
long sum2 = 0;
for (int i = 0; i < 1_000_000; i++) sum2 += RandomNumberGenerator.GetInt32(1, 7);
long t2 = sw.ElapsedMilliseconds;

Console.WriteLine($"\nSystem.Random:          1 млн бросков за {t1} мс");
Console.WriteLine($"RandomNumberGenerator:  1 млн бросков за {t2} мс");

Криптостойкий генератор в разы медленнее, поэтому для игрового процесса его не используют. Но лутбоксы и розыгрыши за реальные деньги считают **на сервере** криптостойким генератором — иначе читеры научатся предсказывать «рандом».

> **В геймдеве.** На Xorshift построен `UnityEngine.Random` в Unity, а в браузере `Math.random()` (движок V8) — его вариант xorshift128+. Такие генераторы выбирают там, где чисел нужно очень много: частицы, шум, процедурная генерация.

---
## Уровень 3. `System.Random` — генератор, который уже есть

### Теория

Писать свой генератор в реальной игре почти никогда не нужно: в C# есть готовый класс `Random`.

| Код | Что даёт |
|---|---|
| `new Random(42)` | генератор с зерном 42: всегда одна и та же последовательность |
| `new Random()` | генератор со случайным зерном |
| `r.Next(10)` | целое от 0 до **9** |
| `r.Next(1, 7)` | целое от 1 до **6** — правая граница **не входит!** |
| `r.NextDouble()` | дробное из `[0, 1)` |

**То же в движках:**

| Unity | Godot (C#) |
|---|---|
| `Random.Range(1, 7)` — целое 1…6 (граница не входит) | `GD.RandRange(1, 6)` — целое 1…6 (граница входит) |
| `Random.Range(1f, 6f)` — дробное, 6 **может** выпасть | `GD.Randf()` — дробное из `[0, 1]` |
| `Random.value` — дробное из `[0, 1]` | |

### Пример 1: одинаковое зерно — одинаковые броски

In [ ]:
var seedA = new Random(42);
var seedB = new Random(42);
Console.WriteLine("Генератор A: " + string.Join(" ", Enumerable.Range(0, 10).Select(_ => seedA.Next(1, 7))));
Console.WriteLine("Генератор B: " + string.Join(" ", Enumerable.Range(0, 10).Select(_ => seedB.Next(1, 7))));

### Пример 2: классическая ошибка

Новичок создаёт генератор **внутри цикла** и в качестве зерна берёт текущее время. Запустите и посмотрите, что получится.

In [ ]:
Console.Write("С ошибкой: ");
for (int i = 0; i < 10; i++)
{
    var r = new Random(DateTime.Now.Millisecond);   // ошибка: новое зерно на каждом шаге
    Console.Write(r.Next(1, 7) + " ");
}
Console.WriteLine();

Console.Write("Правильно: ");
var good = new Random();                            // правильно: один генератор на всё
for (int i = 0; i < 10; i++)
    Console.Write(good.Next(1, 7) + " ");

### Теория: почему нельзя писать `rand() % 6`

В старом коде часто встречается бросок кубика как `1 + rand() % 6`. Проблема в том, что генератор выдаёт фиксированное количество разных чисел, и оно не делится на 6 нацело. Например, `2³² = 6 · 715 827 882 + 4`: остатки 0–3 получаются на один вариант чаще, чем 4 и 5. Для 32-битного генератора перекос крошечный, но если генератор выдаёт мало разных чисел, он огромный. Посмотрите на генератор с восемью вариантами:

In [ ]:
// Генератор, который выдаёт числа 0..7 (8 вариантов), и бросок кубика как v % 6
var bias = new int[6];
for (int v = 0; v < 8; v++) bias[v % 6]++;
for (int i = 0; i < 6; i++)
    Console.WriteLine($"грань {i + 1}: выпадает в {bias[i]} вариантах из 8");

Грани 1 и 2 выпадают вдвое чаще остальных. Метод `r.Next(min, max)` эту проблему решает правильно — пользуйтесь им.

### Задание 3.1

Напишите `RollDie` — бросок кубика с `sides` гранями (d6, d20 и т. д.). Результат должен быть от 1 до `sides` **включительно**.

In [ ]:
int RollDie(Random r, int sides)
{
    // TODO
    throw new NotImplementedException();
}

In [ ]:
Begin("3.1", "RollDie — бросок кубика");
Test("d6: за 60 000 бросков выпали все грани от 1 до 6 и ничего лишнего", () =>
{
    var r = new Random(1);
    var seen = new HashSet<int>();
    for (int i = 0; i < 60_000; i++) seen.Add(RollDie(r, 6));
    return seen.SetEquals(new[] { 1, 2, 3, 4, 5, 6 });
}, "Не выпадает 6? Вспомните: в r.Next(min, max) число max не входит.");
Test("d20: выпадают и 1, и 20", () =>
{
    var r = new Random(2);
    var rolls = Enumerable.Range(0, 20_000).Select(_ => RollDie(r, 20)).ToList();
    return rolls.Min() == 1 && rolls.Max() == 20;
});
Done();

### Задание 3.2

Напишите `Chance` — событие, которое происходит с вероятностью `p`. Например, `Chance(r, 0.15)` — это «крит с шансом 15 %».

Идея: возьмите `r.NextDouble()` — равномерное число из `[0, 1)`. Оно попадает на отрезок `[0, p)` ровно с вероятностью `p`.

In [ ]:
bool Chance(Random r, double p)
{
    // TODO
    throw new NotImplementedException();
}

In [ ]:
Begin("3.2", "Chance — событие с вероятностью p");
Test("p = 0.3 срабатывает примерно в 30 % случаев", () =>
{
    var r = new Random(3);
    int hits = 0;
    for (int i = 0; i < 100_000; i++) if (Chance(r, 0.3)) hits++;
    return Math.Abs(hits / 100_000.0 - 0.3) < 0.01;
});
Test("p = 0 не срабатывает никогда", () =>
{
    var r = new Random(4);
    return Enumerable.Range(0, 10_000).All(_ => !Chance(r, 0));
}, "Используйте строгое сравнение < , а не <=.");
Test("p = 1 срабатывает всегда", () =>
{
    var r = new Random(5);
    return Enumerable.Range(0, 10_000).All(_ => Chance(r, 1));
});
Done();

### Вопросы

Почему в примере 2 «с ошибкой» выпадают одинаковые числа?

*Ваш ответ:* …

---
## Уровень 4. Закон больших чисел

### Теория

Если кубик честный, каждая грань выпадает с вероятностью `1/6 ≈ 0,167`. Но это не значит, что за 6 бросков выпадет каждая грань ровно по разу! Частота приближается к вероятности, только когда бросков **много** — это **закон больших чисел**. Так же мы проверяем, что генератор работает честно.

### Задание 4.1

Напишите `CountRolls`: бросьте d6 `n` раз (используйте свою `RollDie`) и посчитайте, сколько раз выпала каждая грань. Верните массив из 7 элементов: `counts[1]` — сколько раз выпала единица, …, `counts[6]` — шестёрка. Элемент `counts[0]` не используется.

In [ ]:
int[] CountRolls(Random r, int n)
{
    int[] counts = new int[7];
    // TODO: n раз бросьте кубик и увеличьте нужный счётчик
    throw new NotImplementedException();
}

In [ ]:
Begin("4.1", "CountRolls — частоты граней");
Test("массив из 7 элементов, counts[0] = 0", () =>
{
    var c = CountRolls(new Random(1), 600);
    return c.Length == 7 && c[0] == 0;
});
Test("сумма счётчиков равна числу бросков", () => CountRolls(new Random(2), 6000).Sum() == 6000);
Test("при 60 000 бросках каждая грань выпала примерно 10 000 раз (±6 %)", () =>
{
    var c = CountRolls(new Random(3), 60_000);
    return Enumerable.Range(1, 6).All(f => Math.Abs(c[f] - 10_000) < 600);
});
Done();

### Пример: смотрим, как частоты выравниваются

In [ ]:
var lawRng = new Random(2026);
foreach (int n in new[] { 60, 600, 60_000 })
{
    int[] counts = CountRolls(lawRng, n);
    Console.WriteLine($"\nБросков: {n}");
    Histogram(Enumerable.Range(1, 6).Select(f => ($"грань {f}", counts[f] / (double)n)), 30);
}

### Теория: ошибка игрока

Закон больших чисел часто понимают неправильно: «Пять раз выпала решка — значит, сейчас точно выпадет орёл, частота же должна выровняться». Это **ошибка игрока** (gambler's fallacy). Броски **независимы**: монета не помнит прошлых бросков, и шанс орла по-прежнему 50 %. Частота выравнивается не потому, что после серии решек орёл «догоняет», а потому, что одна серия теряется среди миллиона бросков.

### Пример: что выпадает после пяти решек подряд

In [ ]:
var coin = new Random(7);
int situations = 0, headsAfter = 0, tailsInRow = 0;
for (int i = 0; i < 1_000_000; i++)
{
    bool heads = coin.Next(2) == 0;
    if (tailsInRow >= 5)          // перед этим броском было 5+ решек подряд
    {
        situations++;
        if (heads) headsAfter++;
    }
    tailsInRow = heads ? 0 : tailsInRow + 1;
}
Console.WriteLine($"Ситуаций «перед броском 5 решек подряд»: {situations}");
Console.WriteLine($"Из них выпал орёл: {headsAfter / (double)situations:P1}");

### Вопросы

При 60 бросках одна грань может выпасть 4 раза, а другая 13. Генератор сломан?

*Ваш ответ:* …

---
## Уровень 5. Один кубик или несколько?

### Теория

Урон оружия 2–12 можно получить двумя способами: бросить один 11-гранный кубик (или 1d12) — или **сумму двух d6** (2d6). Диапазон почти одинаковый, но ощущения у игрока совершенно разные!

Вспомним характеристики случайной величины:

- **математическое ожидание** `M(X)` — среднее значение;
- **дисперсия** `D(X)` — средний квадрат отклонения от среднего, мера разброса:

$$M = \frac{1}{n}\sum x_i, \qquad D = \frac{1}{n}\sum (x_i - M)^2$$

**Где это встречается.** В настольной игре «Колонизаторы» (Catan) каждый ход бросают 2d6, и каждой клетке с ресурсом приписано число от 2 до 12. Сумма 7 выпадает чаще всех (6 вариантов из 36) — поэтому она и не приносит ресурсов, а двигает разбойника. Клетки с числами 6 и 8 (по 5 вариантов из 36) самые выгодные, клетки с 2 и 12 (по 1 варианту) почти бесполезны.

### Задание 5.1

Напишите `RollSum` — сумму `count` бросков кубика с `sides` гранями. Например, `RollSum(r, 3, 6)` — это 3d6.

In [ ]:
int RollSum(Random r, int count, int sides)
{
    // TODO
    throw new NotImplementedException();
}

In [ ]:
Begin("5.1", "RollSum — сумма кубиков");
Test("3d6 всегда от 3 до 18, и крайние значения встречаются", () =>
{
    var r = new Random(1);
    var rolls = Enumerable.Range(0, 100_000).Select(_ => RollSum(r, 3, 6)).ToList();
    return rolls.Min() == 3 && rolls.Max() == 18;
});
Test("среднее 3d6 ≈ 10,5", () =>
{
    var r = new Random(2);
    double avg = Enumerable.Range(0, 100_000).Select(_ => RollSum(r, 3, 6)).Average();
    return Math.Abs(avg - 10.5) < 0.05;
});
Done();

### Задание 5.2

Напишите `MeanVar`, которая по массиву чисел возвращает среднее и дисперсию по формулам выше. Функция возвращает **кортеж** — два значения сразу: `return (mean, variance);`

In [ ]:
(double Mean, double Variance) MeanVar(int[] data)
{
    double mean = 0;
    double variance = 0;
    // TODO: 1) посчитайте среднее  2) посчитайте средний квадрат отклонения от среднего
    throw new NotImplementedException();
}

In [ ]:
Begin("5.2", "MeanVar — среднее и дисперсия");
Test("{1, 2, 3, 4}: M = 2,5; D = 1,25", () =>
{
    var (m, d) = MeanVar(new[] { 1, 2, 3, 4 });
    return Math.Abs(m - 2.5) < 1e-9 && Math.Abs(d - 1.25) < 1e-9;
}, "Проверьте, что делите на количество элементов как дробное число.");
Test("{5, 5, 5}: M = 5; D = 0", () =>
{
    var (m, d) = MeanVar(new[] { 5, 5, 5 });
    return Math.Abs(m - 5) < 1e-9 && Math.Abs(d) < 1e-9;
});
Done();

### Пример: сравниваем 1d12, 2d6 и 3d6

Теоретические значения: 1d12 — `M = 6,5`, `D ≈ 11,92`; 2d6 — `M = 7`, `D ≈ 5,83`; 3d6 — `M = 10,5`, `D = 8,75`.

In [ ]:
var diceRng = new Random(2026);

void CompareDice(string name, Func<int> roll, int min, int max)
{
    int n = 100_000;
    int[] data = Enumerable.Range(0, n).Select(_ => roll()).ToArray();
    var (mean, variance) = MeanVar(data);
    Console.WriteLine($"\n{name}:  M ≈ {mean:0.00},  D ≈ {variance:0.00}");
    Histogram(Enumerable.Range(min, max - min + 1)
                        .Select(v => (v.ToString(), data.Count(d => d == v) / (double)n)), 30);
}

CompareDice("1d12", () => RollSum(diceRng, 1, 12), 1, 12);
CompareDice("2d6",  () => RollSum(diceRng, 2, 6),  2, 12);
CompareDice("3d6",  () => RollSum(diceRng, 3, 6),  3, 18);

### Задание 5.3 (бонус): бросок с преимуществом

В D&D и многих RPG есть механика **преимущества**: бросаешь d20 два раза и берёшь **больший** результат. Напишите `RollWithAdvantage`. Проверка покажет, как сильно это меняет шансы.

In [ ]:
int RollWithAdvantage(Random r)
{
    // TODO
    throw new NotImplementedException();
}

In [ ]:
Begin("5.3", "RollWithAdvantage — бросок с преимуществом (бонус)");
Test("среднее ≈ 13,8 (у обычного d20 — 10,5)", () =>
{
    var r = new Random(1);
    double avg = Enumerable.Range(0, 100_000).Select(_ => RollWithAdvantage(r)).Average();
    return Math.Abs(avg - 13.825) < 0.06;
});
Test("шанс выбросить 15+ ≈ 51 % (у обычного d20 — 30 %)", () =>
{
    var r = new Random(2);
    double p = Enumerable.Range(0, 100_000).Count(_ => RollWithAdvantage(r) >= 15) / 100_000.0;
    return Math.Abs(p - 0.51) < 0.01;
});
Done();

### Вопросы

1. Какой вариант дать оружию, если нужен **стабильный** урон, а какой — если нужны «качели» (то густо, то пусто)?
2. Шанс 15+ на d20 с преимуществом — `1 − (14/20)²`. Откуда эта формула?

*Ваш ответ:* …

---
## Уровень 6. Лут и редкость

### Теория

В сундуке разные предметы выпадают с разной вероятностью. Дизайнер задаёт каждому исходу **вес**:

| Редкость | Вес | Вероятность |
|---|---|---|
| Обычный | 70 | 70 / 100 = 70 % |
| Редкий | 20 | 20 % |
| Эпический | 8 | 8 % |
| Легендарный | 2 | 2 % |

Вероятность исхода = его вес / сумма всех весов.

**Алгоритм выбора.** Выложим веса на отрезок подряд и «бросим дротик» — случайное число `roll` от 0 до суммы весов:

```
 0                          70        90   98 100
 |========= Обычный =========|= Редкий =|Эп|Л|
                                  ↑ roll = 83 → Редкий
```

Идём по предметам и накапливаем веса: `acc = 70`, потом `90`, потом `98`, потом `100`. Как только `roll < acc` — нашли нужный предмет.

По сути это тот же приём, что и бросок кубика: равномерное число превращаем в исход с нужной вероятностью, просто отрезки разной длины.

### Задание 6.1

Допишите `WeightedChoice`.

In [ ]:
string WeightedChoice(Random r, string[] items, int[] weights)
{
    int total = weights.Sum();
    int roll = r.Next(total);      // случайное число от 0 до total-1
    int acc = 0;
    // TODO: идите по предметам, прибавляйте вес к acc;
    //       как только roll < acc — верните этот предмет
    throw new NotImplementedException();
}

In [ ]:
Begin("6.1", "WeightedChoice — взвешенный выбор");
Test("веса 70/20/8/2 дают частоты 70 %/20 %/8 %/2 % (±1 %)", () =>
{
    var r = new Random(1);
    string[] items = { "О", "Р", "Э", "Л" };
    int[] w = { 70, 20, 8, 2 };
    var cnt = new Dictionary<string, int>();
    for (int i = 0; i < 100_000; i++)
    {
        string it = WeightedChoice(r, items, w);
        cnt[it] = cnt.GetValueOrDefault(it) + 1;
    }
    for (int i = 0; i < items.Length; i++)
        if (Math.Abs(cnt.GetValueOrDefault(items[i]) / 100_000.0 - w[i] / 100.0) > 0.01) return false;
    return true;
}, "Сравнение должно быть строгим: roll < acc.");
Test("предмет с весом 0 не выпадает никогда", () =>
{
    var r = new Random(2);
    string[] items = { "a", "b", "c" };
    return Enumerable.Range(0, 10_000).All(_ => WeightedChoice(r, items, new[] { 5, 0, 5 }) != "b")
        && Enumerable.Range(0, 10_000).All(_ => WeightedChoice(r, items, new[] { 5, 5, 0 }) != "c");
});
Test("веса 1 и 1 — примерно 50 на 50", () =>
{
    var r = new Random(3);
    int a = Enumerable.Range(0, 100_000).Count(_ => WeightedChoice(r, new[] { "a", "b" }, new[] { 1, 1 }) == "a");
    return Math.Abs(a / 100_000.0 - 0.5) < 0.01;
});
Done();

### Пример: открываем 10 000 сундуков

In [ ]:
string[] rarities = { "Обычный", "Редкий", "Эпический", "Легендарный" };
int[] rarityWeights = { 70, 20, 8, 2 };
var lootRng = new Random(2026);
var drops = new Dictionary<string, int>();
for (int i = 0; i < 10_000; i++)
{
    string item = WeightedChoice(lootRng, rarities, rarityWeights);
    drops[item] = drops.GetValueOrDefault(item) + 1;
}

Console.WriteLine("Редкость       ожидали   выпало");
for (int i = 0; i < rarities.Length; i++)
    Console.WriteLine($"{rarities[i],-12} {rarityWeights[i] * 100,9} {drops.GetValueOrDefault(rarities[i]),8}");

### Вопросы

1. Как сделать легендарку **в два раза реже**, не трогая веса остальных предметов? Какой станет её вероятность точно?
2. Почему «ожидали» и «выпало» не совпадают до единицы?

*Ваш ответ:* …

> **В геймдеве.** Это «таблицы лута» почти в любой RPG. Обычно делают два шага: сначала взвешенно выбирают **редкость**, потом равновероятно — предмет этой редкости. Тогда новый предмет можно добавить, не пересчитывая все веса.

---
## Уровень 7. Перемешиваем колоду

### Теория

Как перемешать колоду, чтобы **все** порядки карт были равновероятны? Правильный способ — **алгоритм Фишера — Йетса**:

> Идём с конца массива. На каждом шаге `i` выбираем случайную позицию `j` от `0` до `i` **включительно** и меняем местами карты `i` и `j`.

Обмен двух элементов в C# удобно писать так: `(a[i], a[j]) = (a[j], a[i]);`

Алгоритм придумали статистики Рональд Фишер и Фрэнк Йетс в 1938 году — для перемешивания вручную, с таблицей случайных чисел и карандашом. Компьютерный вариант описал Ричард Дурстенфельд в 1964 году. В .NET 8 появился готовый метод `Random.Shuffle(array)` — внутри у него тот же алгоритм. В реальном проекте пользуйтесь готовым методом, а сейчас напишем его сами, чтобы понять, почему «очевидный» способ не работает.

### Пример: как НЕ надо делать

Новичок меняет каждую карту с **любой** случайной картой. Выглядит разумно — но перемешаем колоду из трёх карт 60 000 раз и посмотрим, как часто выпадает каждый порядок. Если бы всё было честно, каждый столбик был бы `1/6 ≈ 0,167`.

In [ ]:
void NaiveShuffle(Random r, int[] deck)
{
    for (int i = 0; i < deck.Length; i++)
    {
        int j = r.Next(deck.Length);           // ошибка: любая позиция
        (deck[i], deck[j]) = (deck[j], deck[i]);
    }
}

Dictionary<string, int> CountPermutations(Action<Random, int[]> shuffle, int trials)
{
    var r = new Random(1);
    var counts = new Dictionary<string, int>();
    for (int t = 0; t < trials; t++)
    {
        int[] deck = { 1, 2, 3 };
        shuffle(r, deck);
        string key = string.Join(" ", deck);
        counts[key] = counts.GetValueOrDefault(key) + 1;
    }
    return counts;
}

var naive = CountPermutations(NaiveShuffle, 60_000);
Histogram(naive.OrderBy(kv => kv.Key).Select(kv => (kv.Key, kv.Value / 60_000.0)), 30);

### Задание 7.1

Напишите `Shuffle` по алгоритму Фишера — Йетса.

In [ ]:
void Shuffle(Random r, int[] deck)
{
    // TODO: for (int i = deck.Length - 1; i > 0; i--) { ... }
    throw new NotImplementedException();
}

In [ ]:
Begin("7.1", "Shuffle — Фишер — Йетс");
Test("после перемешивания в колоде те же 52 карты", () =>
{
    int[] deck = Enumerable.Range(0, 52).ToArray();
    Shuffle(new Random(1), deck);
    return deck.OrderBy(v => v).SequenceEqual(Enumerable.Range(0, 52));
});
Test("колода действительно перемешивается", () =>
{
    var r = new Random(2);
    int same = 0;
    for (int t = 0; t < 100; t++)
    {
        int[] deck = Enumerable.Range(0, 52).ToArray();
        Shuffle(r, deck);
        if (deck.SequenceEqual(Enumerable.Range(0, 52))) same++;
    }
    return same <= 1;
});
Test("все 6 порядков трёх карт равновероятны", () =>
{
    var counts = CountPermutations(Shuffle, 60_000);
    return counts.Count == 6 && counts.Values.All(v => Math.Abs(v / 60_000.0 - 1.0 / 6) < 0.012);
}, "Позиция j должна выбираться от 0 до i включительно: r.Next(i + 1).");
Done();

Histogram(CountPermutations(Shuffle, 60_000).OrderBy(kv => kv.Key).Select(kv => (kv.Key, kv.Value / 60_000.0)), 30);

### «Мешок» (shuffle bag)

Иногда игроку обидно, когда нужная вещь не выпадает очень долго. Решение — **мешок**: кладём в него все исходы, перемешиваем и достаём **по одному без возвращения**. Мешок опустел — наполняем и перемешиваем заново.

Так работает современный Тетрис («7-bag»): каждые 7 фигур — это все 7 разных фигур в случайном порядке. Между двумя «палками» может оказаться не больше 12 других фигур (если палка была первой в одном мешке и последней в следующем). В старых версиях Тетриса фигуры выбирались независимо, и игроки могли ждать палку по 30–40 ходов.

Тот же приём используют для музыки в плеере (чтобы песни не повторялись подряд), для реплик NPC и для порядка случайных событий.

### Задание 7.2

Допишите метод `Next` в классе `ShuffleBag`.

In [ ]:
class ShuffleBag
{
    private readonly char[] items;
    private readonly Random rng;
    private readonly List<char> bag = new List<char>();

    public ShuffleBag(string items, Random rng)
    {
        this.items = items.ToCharArray();
        this.rng = rng;
    }

    public char Next()
    {
        // TODO:
        // 1) если мешок пуст (bag.Count == 0): добавьте в него все items (bag.AddRange)
        //    и перемешайте его по Фишеру — Йетсу (используйте rng)
        // 2) возьмите последний элемент мешка, удалите его (bag.RemoveAt) и верните
        throw new NotImplementedException();
    }
}

In [ ]:
Begin("7.2", "ShuffleBag — мешок как в Тетрисе");
Test("каждые 7 фигур — это все 7 разных фигур", () =>
{
    var bag = new ShuffleBag("IJLOSTZ", new Random(1));
    for (int block = 0; block < 100; block++)
    {
        var seven = new string(Enumerable.Range(0, 7).Select(_ => bag.Next()).OrderBy(ch => ch).ToArray());
        if (seven != "IJLOSTZ") return false;
    }
    return true;
});
Test("порядок внутри мешков разный", () =>
{
    var bag = new ShuffleBag("IJLOSTZ", new Random(2));
    var blocks = Enumerable.Range(0, 20)
        .Select(_ => new string(Enumerable.Range(0, 7).Select(__ => bag.Next()).ToArray()))
        .ToList();
    return blocks.Distinct().Count() > 10;
}, "Не забудьте перемешать мешок после заполнения.");
Done();

var tetris = new ShuffleBag("IJLOSTZ", new Random(2026));
for (int block = 0; block < 4; block++)
    Console.Write(new string(Enumerable.Range(0, 7).Select(_ => tetris.Next()).ToArray()) + "  ");

### Вопросы

1. Почему у «новичка» порядки выпадают неравномерно? Подсказка: сколько всего вариантов выбрать три числа `j`, и сколько существует порядков трёх карт?
2. Чем мешок хуже честного случайного выбора? Подумайте, что может заметить внимательный игрок в Тетрисе.

*Ваш ответ:* …

---
## Уровень 8. Честно, но обидно: псевдослучайное распределение (PRD)

### Теория

Игроки плохо чувствуют вероятность. Три промаха подряд при шансе попадания 75 % кажутся «нечестными», хотя это случается в 1,6 % случаев — то есть постоянно, если игроков миллионы.

В Warcraft III и Dota 2 шанс крита **растёт после каждой неудачи**:

$$P(N) = C \cdot N$$

где `N` — номер попытки после последнего крита. Сработал крит — `N` сбрасывается в 0. Константу `C` подбирают так, чтобы **в среднем** частота осталась заявленной:

| Заявленный шанс | C | Гарантированный успех на попытке |
|---|---|---|
| 10 % | 0,0148 | 68 |
| 20 % | 0,0557 | 18 |
| 25 % | 0,0847 | 12 |
| 30 % | 0,1189 | 9 |
| 50 % | 0,3021 | 4 |

Обратите внимание: `C` заметно меньше заявленного шанса. Первый удар после крита срабатывает всего с вероятностью 8,5 %, зато каждый следующий — всё вероятнее. Так в Dota 2 работают криты, оглушения и уклонения.

### Задание 8.1

Допишите метод `Roll` в классе `PrdCrit`.

In [ ]:
class PrdCrit
{
    private readonly double c;
    private readonly Random rng;
    private int attempt = 0;          // номер попытки после последнего крита

    public PrdCrit(double c, Random rng)
    {
        this.c = c;
        this.rng = rng;
    }

    public bool Roll()
    {
        // TODO:
        // 1) увеличьте attempt на 1
        // 2) текущий шанс = c * attempt
        // 3) если rng.NextDouble() < шанс: сбросьте attempt в 0 и верните true
        // 4) иначе верните false
        throw new NotImplementedException();
    }
}

In [ ]:
Begin("8.1", "PrdCrit — крит с растущим шансом");
Test("в среднем крит срабатывает в 25 % ударов", () =>
{
    var crit = new PrdCrit(0.0847, new Random(1));
    int hits = 0;
    for (int i = 0; i < 200_000; i++) if (crit.Roll()) hits++;
    return Math.Abs(hits / 200_000.0 - 0.25) < 0.006;
}, "Частота слишком большая? Скорее всего, вы забыли сбросить attempt после крита.");
Test("за 200 000 ударов нет серии промахов длиннее 11", () =>
{
    var crit = new PrdCrit(0.0847, new Random(2));
    int streak = 0, longest = 0;
    for (int i = 0; i < 200_000; i++)
    {
        if (crit.Roll()) streak = 0;
        else longest = Math.Max(longest, ++streak);
    }
    return longest <= 11;
});
Done();

### Пример: честный рандом против PRD

In [ ]:
int LongestMissStreak(Func<bool> roll, int attacks)
{
    int streak = 0, longest = 0;
    for (int i = 0; i < attacks; i++)
    {
        if (roll()) streak = 0;
        else longest = Math.Max(longest, ++streak);
    }
    return longest;
}

var fairRng = new Random(3);
var prd = new PrdCrit(0.0847, new Random(3));
Console.WriteLine($"Честные 25 %: самая длинная серия промахов за 100 000 ударов — {LongestMissStreak(() => Chance(fairRng, 0.25), 100_000)}");
Console.WriteLine($"PRD:          самая длинная серия промахов за 100 000 ударов — {LongestMissStreak(prd.Roll, 100_000)}");

### Вопросы

1. Почему при PRD серия промахов **никогда** не бывает длиннее 11? Посчитайте `C · 12`.
2. Какой минус у PRD? Подумайте, может ли опытный игрок этим воспользоваться.

*Ваш ответ:* …

---
## Уровень 9. Метод Монте-Карло

### Теория

Иногда вероятность трудно (или лень) считать по формуле. Тогда можно **сыграть миллион раз** и посчитать частоту:

$$P(A) \approx \frac{\text{сколько раз произошло } A}{\text{сколько всего опытов}}$$

Это и есть **метод Монте-Карло**.

**Немного истории.** В 1946 году математик Станислав Улам выздоравливал после болезни и раскладывал пасьянсы. Ему стало интересно, с какой вероятностью пасьянс сходится. Посчитать это комбинаторикой оказалось почти невозможно, и Улам понял: проще разложить пасьянс сто раз и посчитать успехи. Вместе с Джоном фон Нейманом он применил эту идею к расчётам движения нейтронов в Лос-Аламосе — на одном из первых компьютеров ENIAC. Секретному методу нужно было кодовое имя; коллега Николас Метрополис предложил «Монте-Карло» — в честь казино в Монако, где любил играть дядя Улама. Кстати, генератором случайных чисел в тех расчётах служил метод середины квадрата из уровня 1.

Схема метода всегда одна:

1. Написать функцию, которая моделирует **один** опыт (один бой, одно открытие сундуков, одну игру).
2. Повторить её `N` раз.
3. Посчитать частоту или среднее.

**Сколько повторений нужно?** Ошибка уменьшается как `1/√N`: в 10 раз точнее — в 100 раз больше опытов.

| N | ошибка оценки вероятности до |
|---|---|
| 100 | ±10 % |
| 1 000 | ±3 % |
| 10 000 | ±1 % |
| 100 000 | ±0,3 % |

### Пример: найдём число π, бросая точки

Бросаем случайные точки в квадрат 1×1. Внутрь четверти круга радиуса 1 попадает доля точек, равная отношению площадей: `(π/4) / 1`. Значит, `π ≈ 4 · (точек внутри) / (всего точек)`.

In [ ]:
double EstimatePi(Random r, int n)
{
    int inside = 0;
    for (int i = 0; i < n; i++)
    {
        double x = r.NextDouble(), y = r.NextDouble();
        if (x * x + y * y <= 1) inside++;
    }
    return 4.0 * inside / n;
}

var piRng = new Random(1);
foreach (int n in new[] { 100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000 })
{
    double est = EstimatePi(piRng, n);
    Console.WriteLine($"N = {n,10:N0}   π ≈ {est:0.00000}   ошибка {Math.Abs(est - Math.PI):0.00000}");
}

### Задание 9.1: шанс на легендарку

Легендарка выпадает из сундука с вероятностью `p`. Игрок открывает `chests` сундуков. Какова вероятность получить **хотя бы одну**? Напишите симуляцию: повторите опыт «открыть `chests` сундуков» `trials` раз и верните долю удачных опытов.

Проверка сравнит ваш ответ с формулой схемы Бернулли: `P = 1 − (1 − p)ⁿ`.

In [ ]:
double AtLeastOne(Random r, double p, int chests, int trials)
{
    int success = 0;
    // TODO: trials раз откройте chests сундуков;
    //       если хотя бы в одном выпала легендарка (Chance(r, p)) — success++
    throw new NotImplementedException();
}

In [ ]:
Begin("9.1", "AtLeastOne — симуляция сундуков");
Test("p = 2 %, 10 сундуков: симуляция совпала с формулой (0,1829)", () =>
{
    double sim = AtLeastOne(new Random(1), 0.02, 10, 200_000);
    double exact = 1 - Math.Pow(1 - 0.02, 10);
    Console.WriteLine($"      симуляция {sim:0.0000}, формула {exact:0.0000}");
    return Math.Abs(sim - exact) < 0.005;
});
Test("p = 10 %, 5 сундуков: симуляция совпала с формулой (0,4095)", () =>
{
    double sim = AtLeastOne(new Random(2), 0.1, 5, 200_000);
    return Math.Abs(sim - (1 - Math.Pow(0.9, 5))) < 0.005;
});
Done();

### Пример: сколько сундуков нужно открыть?

В среднем легендарка выпадает раз в `1/p = 50` сундуков. Но какой шанс её получить, открыв ровно 50? Посмотрите на график.

In [ ]:
double pLeg = 0.02;
Histogram(Enumerable.Range(1, 12).Select(k => k * 20)
          .Select(n => ($"{n} сундуков", 1 - Math.Pow(1 - pLeg, n))), 40);
Console.WriteLine($"\nЗа 50 сундуков: {1 - Math.Pow(1 - pLeg, 50):P1}");
Console.WriteLine($"Для 90 % гарантии нужно {Math.Ceiling(Math.Log(0.1) / Math.Log(1 - pLeg))} сундуков");

### Задание 9.2: добавим гарант

Каждый третий игрок не получит легендарку за «средние» 50 сундуков — и будет недоволен. Поэтому в играх делают **гарант** (pity): если за `pity − 1` сундуков легендарка не выпала, `pity`-й сундук выдаёт её обязательно.

Напишите `ChestsUntilLegendary`: открывайте сундуки, пока не выпадет легендарка, и верните номер сундука, на котором она выпала.

Посчитать среднее с гарантом по формуле уже неудобно — а симуляцией легко.

In [ ]:
int ChestsUntilLegendary(Random r, double p, int pity)
{
    // TODO: для k = 1, 2, 3, ...
    //   если k == pity — гарант сработал, верните k
    //   если выпала легендарка (Chance(r, p)) — верните k
    throw new NotImplementedException();
}

In [ ]:
Begin("9.2", "ChestsUntilLegendary — гарант");
Test("с гарантом на 50-м сундуке — никогда больше 50", () =>
{
    var r = new Random(1);
    return Enumerable.Range(0, 50_000).All(_ => ChestsUntilLegendary(r, 0.02, 50) <= 50);
});
Test("при p = 1 легендарка сразу в первом сундуке", () =>
{
    var r = new Random(2);
    return Enumerable.Range(0, 1000).All(_ => ChestsUntilLegendary(r, 1.0, 50) == 1);
});
Test("в среднем ≈ 31,8 сундука (p = 2 %, гарант 50)", () =>
{
    var r = new Random(3);
    double avg = Enumerable.Range(0, 100_000).Select(_ => ChestsUntilLegendary(r, 0.02, 50)).Average();
    return Math.Abs(avg - 31.79) < 0.3;
}, "Гарант срабатывает на сундуке номер pity, а не pity + 1.");
Done();

In [ ]:
var chestRng = new Random(5);
int T = 100_000;
var noPity   = Enumerable.Range(0, T).Select(_ => ChestsUntilLegendary(chestRng, 0.02, int.MaxValue)).ToArray();
var withPity = Enumerable.Range(0, T).Select(_ => ChestsUntilLegendary(chestRng, 0.02, 50)).ToArray();

Console.WriteLine($"Без гаранта: в среднем {noPity.Average():0.0} сундука, самый невезучий — {noPity.Max()}");
Console.WriteLine($"С гарантом:  в среднем {withPity.Average():0.0} сундука, самый невезучий — {withPity.Max()}");
Console.WriteLine($"Гарант сработал у {withPity.Count(v => v == 50) * 100.0 / T:0.0} % игроков");
Console.WriteLine($"Реальная частота легендарок с гарантом: {1 / withPity.Average():P1} (а не 2 %)");

### Задание 9.3: задача коллекционера

В каждом наборе — одна случайная фигурка из `n` (все равновероятны). Сколько наборов нужно купить, чтобы собрать **всю** коллекцию? Напишите функцию для **одного** коллекционера; проверка сама повторит её много раз.

Подсказка: удобно хранить собранные фигурки в `HashSet<int>` — он не хранит повторы, а `set.Count` покажет, сколько разных уже есть.

In [ ]:
int BoxesToCollect(Random r, int n)
{
    var got = new HashSet<int>();
    int boxes = 0;
    // TODO: пока собрано меньше n разных фигурок — покупаем набор
    throw new NotImplementedException();
}

In [ ]:
Begin("9.3", "BoxesToCollect — коллекционер");
Test("коллекция из 1 фигурки собирается за 1 набор", () => BoxesToCollect(new Random(1), 1) == 1);
Test("никогда не меньше n наборов", () =>
{
    var r = new Random(2);
    return Enumerable.Range(0, 10_000).All(_ => BoxesToCollect(r, 6) >= 6);
});
Test("для 6 фигурок в среднем ≈ 14,7 набора", () =>
{
    var r = new Random(3);
    double avg = Enumerable.Range(0, 50_000).Select(_ => BoxesToCollect(r, 6)).Average();
    return Math.Abs(avg - 14.7) < 0.15;
});
Done();

var colRng = new Random(2026);
var boxes = Enumerable.Range(0, 100_000).Select(_ => BoxesToCollect(colRng, 6)).ToArray();
Console.WriteLine($"\nУложились в 10 наборов: {boxes.Count(b => b <= 10) / 100_000.0:P0}");
Console.WriteLine($"Уложились в 20 наборов: {boxes.Count(b => b <= 20) / 100_000.0:P0}");
Console.WriteLine($"Самому невезучему понадобилось: {boxes.Max()}");

### Вопросы

1. С гарантом средняя частота легендарок стала около 3,1 % вместо 2 %. Почему это важно знать дизайнеру **до** выхода игры?
2. Почему последнюю фигурку из шести приходится ждать дольше всего?

*Ваш ответ:* …

> **В геймдеве.** Монте-Карло — главный инструмент баланса: прогнать 10 000 боёв и узнать процент побед каждого класса, проверить экономику, найти сложность уровня. На этом же принципе работает ИИ на основе поиска по дереву Монте-Карло (MCTS): он много раз доигрывает партию случайными ходами и выбирает ход с лучшей статистикой.

---
## Уровень 10. Процедурная генерация: пещера

### Теория

Как сгенерировать уникальную пещеру для «рогалика»? Самый простой способ — **случайное блуждание** («пьяный шахтёр»):

1. Карта целиком из камня, шахтёр стоит в центре.
2. На каждом шаге он идёт в случайную сторону (вверх, вниз, влево, вправо) и превращает клетку в пол.
3. Края карты трогать нельзя — там всегда стена.

Тот же `seed` даёт ту же пещеру — так игроки могут делиться понравившимися картами.

**Откуда это пошло.** Игра Rogue (1980) генерировала новое подземелье при каждом прохождении — от неё жанр и получил название «рогалик» (roguelike). Сегодня процедурная генерация есть в Minecraft, Terraria, Spelunky, Hades, No Man's Sky. Для ландшафтов вместо блуждания обычно берут **шум Перлина**: Кен Перлин придумал его для фильма «Трон» (1982) и в 1997 году получил за него технический «Оскар». Шум Перлина — это тоже ГПСЧ, только сглаженный: соседние точки получают близкие значения, и вместо «снега» получаются холмы и горы.

### Задание 10.1

Допишите тело цикла в `DigCave`.

In [ ]:
bool[,] DigCave(Random r, int width, int height, int steps)
{
    var floor = new bool[width, height];     // false — камень, true — пол
    int x = width / 2, y = height / 2;       // шахтёр стартует в центре
    floor[x, y] = true;
    int[] dx = { 1, -1, 0, 0 };              // 4 направления:
    int[] dy = { 0, 0, 1, -1 };              // вправо, влево, вниз, вверх

    for (int s = 0; s < steps; s++)
    {
        // TODO:
        // 1) выберите случайное направление: int d = r.Next(4);
        // 2) посчитайте новую клетку: nx = x + dx[d], ny = y + dy[d]
        // 3) если она НЕ на краю карты (1 <= nx < width - 1 и 1 <= ny < height - 1):
        //    перейдите в неё (x = nx; y = ny;) и сделайте её полом
    }
    return floor;
}

In [ ]:
void PrintCave(bool[,] cave)
{
    for (int y = 0; y < cave.GetLength(1); y++)
    {
        var line = new StringBuilder();
        for (int x = 0; x < cave.GetLength(0); x++)
            line.Append(cave[x, y] ? "  " : "██");
        Console.WriteLine(line.ToString());
    }
}

bool SameCave(bool[,] a, bool[,] b) => a.Cast<bool>().SequenceEqual(b.Cast<bool>());

Begin("10.1", "DigCave — генерация пещеры");
Func<bool[,]> testCave = () => DigCave(new Random(8), 40, 20, 700);
Test("края карты остались камнем", () =>
{
    var c = testCave();
    for (int x = 0; x < 40; x++) if (c[x, 0] || c[x, 19]) return false;
    for (int y = 0; y < 20; y++) if (c[0, y] || c[39, y]) return false;
    return true;
}, "Проверьте условие: 1 <= nx < width - 1 и 1 <= ny < height - 1.");
Test("шахтёр накопал больше 50 клеток", () => testCave().Cast<bool>().Count(v => v) > 50,
     "Не забудьте переместить шахтёра (x = nx; y = ny;) и отметить клетку полом.");
Test("тот же seed — та же пещера", () =>
    SameCave(DigCave(new Random(5), 40, 20, 700), DigCave(new Random(5), 40, 20, 700)));
Test("другой seed — другая пещера", () =>
    !SameCave(DigCave(new Random(5), 40, 20, 700), DigCave(new Random(6), 40, 20, 700)));
Done();

Console.WriteLine();
PrintCave(DigCave(new Random(8), 40, 20, 700));

### Эксперимент

Поменяйте `seed` и число шагов и посмотрите, как меняется пещера. Попробуйте 200, 700 и 3000 шагов.

In [ ]:
int caveSeed = 42;
int caveSteps = 1500;
PrintCave(DigCave(new Random(caveSeed), 40, 20, caveSteps));

### Вопросы

Как изменить алгоритм, чтобы получалась не одна «клякса», а несколько залов, соединённых коридорами?

*Ваш ответ:* …

---
## Финальный босс: настраиваем «Рыбалку»

Вы — геймдизайнер мини-игры «Рыбалка». На каждый заброс удочки ловится одна рыба:

| Рыба | Вес |
|---|---|
| Ёрш | 600 |
| Окунь | 250 |
| Щука | 120 |
| **Золотая рыбка** | **?** |

Ещё есть **гарант**: если 39 забросов подряд не было золотой рыбки, 40-й заброс приносит её обязательно.

**Требование продюсера:** игрок должен ловить золотую рыбку **в среднем раз в 25 забросов** (допустимо от 24,4 до 25,6).

**Ваша задача** — подобрать вес золотой рыбки (целое число) методом Монте-Карло.

### Шаг 1. Один игрок

Напишите `CastsUntilGolden`: игрок забрасывает удочку, пока не поймает золотую рыбку. Верните номер заброса. Используйте `WeightedChoice` из уровня 6 и не забудьте про гарант.

### Шаг 2. Много игроков

Напишите `AverageCasts`: повторите `CastsUntilGolden` `trials` раз и верните среднее.

In [ ]:
string[] fish = { "Ёрш", "Окунь", "Щука", "Золотая рыбка" };

int CastsUntilGolden(Random r, int goldenWeight)
{
    int[] weights = { 600, 250, 120, goldenWeight };
    // TODO
    throw new NotImplementedException();
}

double AverageCasts(int goldenWeight, int trials)
{
    var r = new Random(123);
    // TODO
    throw new NotImplementedException();
}

### Шаг 3. Ищем вес

Запустите перебор. Найдите, между какими весами среднее проходит через 25, и уточните — меняйте шаг и границы цикла.

In [ ]:
for (int w = 5; w <= 60; w += 5)
    Console.WriteLine($"вес {w,3}  →  в среднем {AverageCasts(w, 20_000):0.00} забросов");

### Шаг 4. Ваш ответ

Впишите найденный вес и запустите проверку.

In [ ]:
int myGoldenWeight = 0;   // TODO: впишите подобранный вес

In [ ]:
double ExactAverage(int w)   // точная формула — чтобы проверка не зависела от случайности
{
    double p = w / (970.0 + w), q = 1 - p, e = 0;
    for (int k = 1; k < 40; k++) e += k * p * Math.Pow(q, k - 1);
    return e + 40 * Math.Pow(q, 39);
}

Begin("Б", "Финальный босс: золотая рыбка");
Test("CastsUntilGolden никогда не превышает 40", () =>
{
    var r = new Random(1);
    return Enumerable.Range(0, 20_000).All(_ => CastsUntilGolden(r, 15) <= 40);
});
Test("при весе 0 рыбка приходит только по гаранту — ровно на 40-м забросе", () =>
{
    var r = new Random(2);
    return Enumerable.Range(0, 2_000).All(_ => CastsUntilGolden(r, 0) == 40);
}, "Гарант срабатывает на 40-м забросе, а не на 41-м.");
Test("AverageCasts(10, 50 000) ≈ 33,0", () => Math.Abs(AverageCasts(10, 50_000) - 32.98) < 0.3);
Test("выбранный вес даёт в среднем от 24,4 до 25,6 заброса", () =>
{
    double e = ExactAverage(myGoldenWeight);
    Console.WriteLine($"      вес {myGoldenWeight}: в среднем {e:0.00} заброса");
    return e >= 24.4 && e <= 25.6;
}, "Сузьте перебор вокруг найденного значения и шагайте по 1.");
Done();

### Шаг 5. Отчёт продюсеру

Допишите отчёт (двойной щелчок по этой ячейке, чтобы редактировать):

- **Вес золотой рыбки:** …
- **Шанс поймать золотую рыбку за один заброс без учёта гаранта** (вес / сумма весов): … %
- **Реальная частота с учётом гаранта** (1 / среднее число забросов): … %
- **У какой доли игроков рыбка пришла по гаранту** (посчитайте симуляцией): … %
- **Что будет, если продюсер попросит «раз в 10 забросов»?** …

---
## Типичные ошибки при работе со случайностью

Соберём в одном месте всё, на чём чаще всего ошибаются — и новички, и опытные разработчики.

1. **Генератор внутри цикла.** `new Random(DateTime.Now.Millisecond)` много раз подряд даёт одинаковые числа (уровень 3). Генератор создают **один раз**.
2. **Границы диапазона.** `r.Next(1, 6)` в C# и `Random.Range(1, 6)` в Unity никогда не выдадут 6. При этом `Random.Range(1f, 6f)` для дробных чисел может выдать 6. В Godot `GD.RandRange(1, 6)` включает обе границы.
3. **`rand() % n`.** Даёт перекос в пользу малых значений (уровень 3). Используйте готовые функции диапазона.
4. **Самописное перемешивание.** Без алгоритма Фишера — Йетса колода перемешивается неравномерно (уровень 7).
5. **Обычный генератор для безопасности.** Для паролей, токенов и денег — только `RandomNumberGenerator` (уровень 2).
6. **Один генератор на всё.** Если лут и погода берут числа из одного генератора, то добавление нового дождя меняет весь лут, и воспроизводимость по зерну ломается. Заводите отдельный генератор для каждой системы: `lootRng`, `weatherRng`, `mapRng`.
7. **Ошибка игрока.** «Пять промахов подряд — значит, сейчас точно попаду». Независимые испытания прошлого не помнят (уровень 4). Если дизайнер хочет, чтобы игра «помнила», нужна специальная механика — PRD, мешок или гарант.
8. **Мало прогонов.** 100 симуляций — ещё не баланс: ошибка до ±10 % (уровень 9).

---
## Мой прогресс

Запустите ячейку — она покажет, какие задания засчитаны. Перед сдачей перезапустите ядро и выполните все ячейки сверху (*Restart and Run All*): так вы убедитесь, что тетрадка работает целиком, а не только в вашем текущем сеансе.

In [ ]:
var allTasks = new (string Id, string Name)[]
{
    ("1.1", "LCG: следующий член"),        ("1.2", "LCG: период"),
    ("2.1", "Xorshift32"),                 ("2.2", "Число из [0, 1)"),
    ("3.1", "Бросок кубика"),              ("3.2", "Событие с шансом p"),
    ("4.1", "Закон больших чисел"),        ("5.1", "Сумма кубиков"),
    ("5.2", "Среднее и дисперсия"),        ("6.1", "Взвешенный выбор"),
    ("7.1", "Фишер — Йетс"),               ("7.2", "Мешок"),
    ("8.1", "PRD"),                        ("9.1", "Шанс на легендарку"),
    ("9.2", "Гарант"),                     ("9.3", "Коллекционер"),
    ("10.1", "Генерация пещеры"),          ("Б", "Финальный босс"),
};

int doneCount = 0;
foreach (var (id, name) in allTasks)
{
    bool ok = _progress.TryGetValue(id, out var v) && v;
    if (ok) doneCount++;
    Console.WriteLine($"[{(ok ? "x" : " ")}] {id,-5} {name}");
}
bool bonus = _progress.TryGetValue("5.3", out var bv) && bv;
Console.WriteLine($"[{(bonus ? "x" : " ")}] 5.3   Бонус: бросок с преимуществом");

Console.WriteLine($"\nВыполнено {doneCount} из {allTasks.Length}.");

---
## Что дальше: контрольная точка № 8

Теперь у вас есть все инструменты для учебного проекта. Разбейтесь на команды по 1–5 человек и выберите тему — во всех темах главный инструмент **метод Монте-Карло**:

1. **Балансировка сложности уровней** — найти параметры уровня, при которых большинство игроков проходит этап.
2. **Бой в пошаговой RPG** — сравнить тактики боя (уровни 3, 6, 9 этой тетрадки).
3. **Система улучшения оружия** — найти оптимальную стратегию трат ресурсов (уровень 9).
4. **Генеративный ландшафт** — процедурные карты (уровень 10).
5. **Коллекционная карточная игра** — эффективность колод (уровни 6, 7).
6. **Социальное взаимодействие** — симулятор альянсов и конфликтов.

**Отчёт (10 баллов):** тема и актуальность — 1; метод Монте-Карло в геймдеве — 1; описание модели — 3; листинги кода — 2; результаты и выводы — 2; оформление — 1. Формат PPT или PDF, срок — неделя.